# Precompute PBMC MVP — Geneformer Embeddings

**Author:** Manuel Murillo  
**Date:** 2026-04-16  
**Purpose:** Embed PBMC 3k healthy cells with Geneformer (Helical SDK), compute UMAP on the embedding space, export as parquet to S3, and write provenance to Neon Postgres.  
**Expected runtime:** 15–25 minutes on Colab T4 GPU  

This notebook produces the **real Geneformer embeddings** that replace the PACKET-02a scanpy baseline placeholder. The output parquet has the same schema (`cell_id`, `cell_type`, `umap_1`, `umap_2`, `embedding_0..511`) so no downstream code changes are needed.

In [ ]:
# Cell 1 — Environment setup
!pip install -r requirements-colab.txt -q

In [ ]:
# Cell 2 — Config: env vars, git SHA, S3 + version constants
import os
import subprocess

# Try Colab userdata first, fall back to os.environ
def _get_secret(key: str) -> str:
    """Read from Colab Secrets or os.environ."""
    try:
        from google.colab import userdata  # type: ignore[import-untyped]
        val = userdata.get(key)
        if val:
            return val
    except (ImportError, Exception):
        pass
    val = os.environ.get(key)
    if not val:
        raise RuntimeError(f"Missing required secret: {key}")
    return val

DATABASE_URL = _get_secret("DATABASE_URL")
S3_BUCKET = _get_secret("S3_BUCKET")
S3_REGION = _get_secret("S3_REGION")
AWS_ACCESS_KEY_ID = _get_secret("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = _get_secret("AWS_SECRET_ACCESS_KEY")

# Set AWS creds for boto3
os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY

# Capture git SHA (may fail on Colab — falls back to placeholder)
try:
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
except Exception:
    git_sha = "colab-no-git"

VERSION = "v1"
S3_KEY_PARQUET = f"{VERSION}/pbmc3k/geneformer_embeddings.parquet"
S3_KEY_UMAP_REDUCER = f"{VERSION}/pbmc3k/pbmc_umap_reducer.joblib"

print(f"Database: {DATABASE_URL[:30]}...")
print(f"S3 bucket: {S3_BUCKET} ({S3_REGION})")
print(f"Git SHA: {git_sha}")
print(f"Version: {VERSION}")

## Step 1: Load PBMC 3k

In [ ]:
# Cell 4 — Load PBMC 3k via scanpy and validate
import scanpy as sc

adata = sc.datasets.pbmc3k_processed()
print(f"Shape: {adata.shape}")
assert adata.n_obs == 2638, f"Expected 2638 cells, got {adata.n_obs}"

# Validate expected keys
assert "louvain" in adata.obs.columns, "Missing louvain cell-type labels"
assert "X_umap" in adata.obsm, "Missing precomputed UMAP coordinates"

# Cell-type distribution
print("\nCell-type distribution (louvain):")
print(adata.obs["louvain"].value_counts().to_string())
print(f"\nUnique cell types: {adata.obs['louvain'].nunique()}")

## Step 2: Geneformer embeddings

In [ ]:
# Cell 6 — Initialize Geneformer via Helical SDK and embed PBMC 3k
import torch
from helical.models.geneformer import Geneformer, GeneformerConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Initialize Geneformer with default v1 model (gf-6L-10M-i2048)
model_config = GeneformerConfig(batch_size=10, device=device)
geneformer = Geneformer(configurer=model_config)
print("Geneformer model loaded.")

# The Helical SDK expects gene names in adata.var. pbmc3k_processed stores
# gene symbols in the var index — copy them to a named column.
adata.var["gene_name"] = adata.var_names

# Process data (tokenization + preparation)
dataset = geneformer.process_data(adata, gene_names="gene_name")
print("Data processed for Geneformer.")

# Generate embeddings
embeddings = geneformer.get_embeddings(dataset)
print(f"Embedding shape: {embeddings.shape}")
assert embeddings.shape[0] == 2638, f"Expected 2638 rows, got {embeddings.shape[0]}"

# Store in AnnData obsm slot
adata.obsm["X_geneformer"] = embeddings

# Sanity check: mean and std of embeddings
import numpy as np
print(f"Mean: {np.mean(embeddings):.6f}, Std: {np.std(embeddings):.6f}")
print(f"Embedding dim: {embeddings.shape[1]}")

## Step 3: UMAP

In [ ]:
# Cell 8 — Compute UMAP on Geneformer embeddings + per-cell-type centroids
import numpy as np
from umap import UMAP
import joblib

# Fit UMAP on Geneformer embedding space (not scanpy's precomputed UMAP)
reducer = UMAP(n_neighbors=15, min_dist=0.5, random_state=42)
umap_coords = reducer.fit_transform(adata.obsm["X_geneformer"])
print(f"UMAP shape: {umap_coords.shape}")
assert umap_coords.shape == (2638, 2), f"Expected (2638, 2), got {umap_coords.shape}"

# Store in AnnData
adata.obsm["X_umap_geneformer"] = umap_coords

# Compute per-cell-type centroids in full embedding space (not UMAP space)
centroids = {}
for ct in adata.obs["louvain"].unique():
    mask = adata.obs["louvain"] == ct
    centroids[str(ct)] = np.mean(adata.obsm["X_geneformer"][mask], axis=0)
    print(f"  {ct}: {int(mask.sum())} cells, centroid norm = {np.linalg.norm(centroids[str(ct)]):.4f}")

print(f"\nComputed centroids for {len(centroids)} cell types")

# Persist fitted UMAP reducer for PACKET-02c (disease projection)
joblib.dump(reducer, "/tmp/pbmc_umap_reducer.joblib")
print("Saved UMAP reducer to /tmp/pbmc_umap_reducer.joblib")

## Step 4: Export

In [ ]:
# Cell 10 — Build DataFrame, write parquet, upload to S3
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import boto3

# Build DataFrame with exact PACKET-02a schema (cell_id, cell_type, umap_1, umap_2, embedding_0..N)
emb_dim = adata.obsm["X_geneformer"].shape[1]
umap_gf = adata.obsm["X_umap_geneformer"]
gf_emb = adata.obsm["X_geneformer"]

base = pd.DataFrame({
    "cell_id": adata.obs_names.astype(str),
    "cell_type": adata.obs["louvain"].astype(str),
    "umap_1": umap_gf[:, 0].astype("float32"),
    "umap_2": umap_gf[:, 1].astype("float32"),
})

emb_df = pd.DataFrame(
    gf_emb.astype(np.float32),
    columns=[f"embedding_{i}" for i in range(emb_dim)],
)

# Align indices before concat (same fix as PACKET-02a)
base = base.reset_index(drop=True)
emb_df = emb_df.reset_index(drop=True)
df = pd.concat([base, emb_df], axis=1)

expected_cols = 4 + emb_dim  # cell_id, cell_type, umap_1, umap_2 + embedding columns
assert df.shape == (2638, expected_cols), f"Expected (2638, {expected_cols}), got {df.shape}"
print(f"DataFrame shape: {df.shape}")

# Write local parquet
local_parquet = "/tmp/geneformer_embeddings.parquet"
table = pa.Table.from_pandas(df, preserve_index=False)
pq.write_table(table, local_parquet, compression="snappy")
import os as _os
size_mb = _os.path.getsize(local_parquet) / (1024 * 1024)
print(f"Parquet written: {local_parquet} ({size_mb:.2f} MB)")
assert size_mb < 10, f"Parquet size {size_mb:.2f} MB exceeds 10 MB guard"

# Upload parquet to S3
s3 = boto3.client("s3", region_name=S3_REGION)
s3.upload_file(local_parquet, S3_BUCKET, S3_KEY_PARQUET)
print(f"Uploaded: s3://{S3_BUCKET}/{S3_KEY_PARQUET} ({size_mb:.2f} MB)")

# Upload fitted UMAP reducer for PACKET-02c
reducer_path = "/tmp/pbmc_umap_reducer.joblib"
s3.upload_file(reducer_path, S3_BUCKET, S3_KEY_UMAP_REDUCER)
reducer_size = _os.path.getsize(reducer_path) / (1024 * 1024)
print(f"Uploaded: s3://{S3_BUCKET}/{S3_KEY_UMAP_REDUCER} ({reducer_size:.2f} MB)")

# Reminder for local fallback copy
print("\n>> Post-run: cp /tmp/geneformer_embeddings.parquet backend/data/parquet/pbmc3k/")
print("   This overwrites the PACKET-02a placeholder with real Geneformer embeddings.")

## Step 5: Provenance

In [ ]:
# Cell 12 — Write precompute_runs row to Neon Postgres
import asyncio
import ssl
from urllib.parse import parse_qs, urlencode, urlparse, urlunparse

from sqlalchemy.ext.asyncio import async_sessionmaker, create_async_engine
from sqlmodel import select
from sqlmodel.ext.asyncio.session import AsyncSession as SQLModelAsyncSession


def _prepare_asyncpg_url(raw_url: str) -> tuple:
    """Strip libpq params from Neon URL for asyncpg compatibility."""
    connect_args = {"statement_cache_size": 0}
    parsed = urlparse(raw_url)
    params = parse_qs(parsed.query, keep_blank_values=True)
    sslmode = params.pop("sslmode", [None])[0]
    if sslmode and sslmode in ("require", "verify-ca", "verify-full"):
        connect_args["ssl"] = ssl.create_default_context()
    params.pop("channel_binding", None)
    clean_query = urlencode({k: v[0] for k, v in params.items()}, doseq=False)
    clean_url = urlunparse(parsed._replace(query=clean_query))
    return clean_url, connect_args


async def _write_provenance():
    """Insert one precompute_runs row for Geneformer on PBMC 3k."""
    # Import models from the backend (requires backend on sys.path or inline definition)
    # In Colab we define the model inline to avoid cloning the full repo.
    import uuid
    from datetime import datetime, timezone
    from typing import Optional
    from sqlmodel import Field, SQLModel

    class Dataset(SQLModel, table=True):
        __tablename__ = "datasets"
        id: Optional[uuid.UUID] = Field(default_factory=uuid.uuid4, primary_key=True)
        slug: str = Field(index=True)
        display_name: str = ""
        citation: str = ""
        license: str = ""
        cell_count: int = 0
        gene_count: int = 0

    class PrecomputeRun(SQLModel, table=True):
        __tablename__ = "precompute_runs"
        id: Optional[uuid.UUID] = Field(default_factory=uuid.uuid4, primary_key=True)
        dataset_id: uuid.UUID = Field(foreign_key="datasets.id", index=True)
        model_name: str
        model_version: str
        parameters: dict = Field(default_factory=dict)
        output_parquet_key: str
        git_sha: str
        created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))

    database_url, connect_args = _prepare_asyncpg_url(DATABASE_URL)
    engine = create_async_engine(database_url, echo=False, connect_args=connect_args)
    session_factory = async_sessionmaker(
        engine, class_=SQLModelAsyncSession, expire_on_commit=False
    )

    try:
        async with session_factory() as session:
            result = await session.exec(select(Dataset).where(Dataset.slug == "pbmc3k"))
            dataset = result.first()
            if dataset is None or dataset.id is None:
                raise RuntimeError(
                    "No datasets row with slug=pbmc3k — run PACKET-01b seed first."
                )

            row = PrecomputeRun(
                dataset_id=dataset.id,
                model_name="geneformer",
                model_version="v1",
                parameters={
                    "embedding_dim": embeddings.shape[1],
                    "max_input_genes": 2048,
                },
                output_parquet_key=S3_KEY_PARQUET,
                git_sha=git_sha,
            )
            session.add(row)
            await session.commit()
            await session.refresh(row)
            print(f"Inserted precompute_runs id={row.id}")
            print(f"  model_name={row.model_name}, model_version={row.model_version}")
            print(f"  output_parquet_key={row.output_parquet_key}")
            print(f"  git_sha={row.git_sha}")
    finally:
        await engine.dispose()


# Colab runs in an existing event loop — use nest_asyncio or await directly
try:
    import nest_asyncio
    nest_asyncio.apply()
    asyncio.get_event_loop().run_until_complete(_write_provenance())
except ImportError:
    # If nest_asyncio not available, try await (works in IPython 8+)
    await _write_provenance()  # type: ignore[top-level-await]

## Verification

In [ ]:
# Cell 14 — Re-read parquet from S3, assert shape, print sample rows + provenance
import pyarrow.parquet as pq
import pyarrow.fs as pafs
import numpy as np

# Re-read from S3
s3_fs = pafs.S3FileSystem(region=S3_REGION)
s3_path = f"{S3_BUCKET}/{S3_KEY_PARQUET}"
table = pq.read_table(s3_path, filesystem=s3_fs)

print(f"Parquet shape from S3: ({table.num_rows}, {table.num_columns})")
assert table.num_rows == 2638, f"Expected 2638 rows, got {table.num_rows}"
assert table.num_columns == 516, f"Expected 516 columns, got {table.num_columns}"

# Print 3 sample rows
sample_df = table.to_pandas().head(3)
print("\nSample rows (first 3):")
print(sample_df[["cell_id", "cell_type", "umap_1", "umap_2", "embedding_0", "embedding_1"]].to_string())

# Verify embeddings are non-zero (real Geneformer, not PACKET-02a placeholder)
emb_col = table.column("embedding_0").to_numpy()
assert not np.all(emb_col == 0), "embedding_0 is all zeros — still the placeholder, not real Geneformer!"
print("\n\u2705 Embeddings are non-zero (real Geneformer output confirmed)")

# Fetch latest provenance row
print("\n--- Provenance ---")
print("Fetch the latest precompute_runs row from Neon:")

async def _fetch_provenance():
    from sqlalchemy.ext.asyncio import create_async_engine, async_sessionmaker
    from sqlmodel.ext.asyncio.session import AsyncSession as SQLModelAsyncSession
    from sqlmodel import select, SQLModel, Field
    import uuid
    from datetime import datetime
    from typing import Optional

    class PrecomputeRun(SQLModel, table=True):
        __tablename__ = "precompute_runs"
        __table_args__ = {"extend_existing": True}
        id: Optional[uuid.UUID] = Field(default_factory=uuid.uuid4, primary_key=True)
        dataset_id: uuid.UUID = Field(foreign_key="datasets.id", index=True)
        model_name: str
        model_version: str
        parameters: dict = Field(default_factory=dict)
        output_parquet_key: str
        git_sha: str
        created_at: Optional[datetime] = None

    database_url, connect_args = _prepare_asyncpg_url(DATABASE_URL)
    engine = create_async_engine(database_url, echo=False, connect_args=connect_args)
    session_factory = async_sessionmaker(
        engine, class_=SQLModelAsyncSession, expire_on_commit=False
    )
    try:
        async with session_factory() as session:
            result = await session.exec(
                select(PrecomputeRun)
                .where(PrecomputeRun.model_name == "geneformer")
                .order_by(PrecomputeRun.created_at.desc())
            )
            row = result.first()
            if row:
                print(f"  id: {row.id}")
                print(f"  model_name: {row.model_name}")
                print(f"  model_version: {row.model_version}")
                print(f"  parameters: {row.parameters}")
                print(f"  output_parquet_key: {row.output_parquet_key}")
                print(f"  git_sha: {row.git_sha}")
                print(f"  created_at: {row.created_at}")
            else:
                print("  No provenance row found!")
    finally:
        await engine.dispose()

try:
    import nest_asyncio
    nest_asyncio.apply()
    asyncio.get_event_loop().run_until_complete(_fetch_provenance())
except ImportError:
    await _fetch_provenance()  # type: ignore[top-level-await]

print("\n\u2705 All verification checks passed!")
print("\n>> Post-run: cp /tmp/geneformer_embeddings.parquet backend/data/parquet/pbmc3k/")